# 03 Transformation Test

This notebook is used to prototype the transformation logic before converting it into PostgreSQL stored procedures.

For each table, the structure is:

```text
1. Query the raw table
2. Apply the transformation SELECT statement

```

In order to compare the raw output with the transformed output


## 1. Connect to PostgreSQL

This notebook reuses the database connection from `src/db_connection.py`.

In [1]:
import pandas as pd
from pathlib import Path
import sys

BASE_DIR = Path.cwd().parent
SRC_DIR = BASE_DIR / "src"

sys.path.append(str(SRC_DIR))

from db_connection import get_engine

engine = get_engine()

2026-09-16 19:10:21,121 | INFO | db_connection | Database environment variables validated successfully.
2026-09-16 19:10:21,121 | INFO | db_connection | Building database URL for host=localhost, port=5432, database=midterm_1_vic, user=postgres
2026-09-16 19:10:21,122 | INFO | db_connection | Creating SQLAlchemy engine.


## 2. Helper function

This helper runs a SQL query in PostgreSQL and returns the result as a pandas DataFrame.


In [2]:
def run_query(query : str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql_query(query, conn)

# 3. Customers transformation

Business rules:
1. Trim `customer_id` to remove extra whitespace.
2. Standardize `city` using title case (e.g., "london" → "London").
3. Convert `signup_date` to `DATE`


## 3.1 Customers — raw data

First, retrieve the data as it currently exists in the `raw.customers` table.


In [5]:
raw_customers_query="""
SELECT
    customer_id,
    city,
    signup_date,
    source_file
FROM raw.customers
WHERE customer_id IS NOT NULL
ORDER BY customer_id;
"""

raw_customers_df = run_query(raw_customers_query)
raw_customers_df.head(10)

,customer_id,city,signup_date,source_file
0,C0001,Bristol,2022-04-25,customers01.csv
1,C0002,London,2022-10-09,customers01.csv
2,C0003,Manchester,2022-08-17,customers01.csv
3,C0004,Manchester,2022-04-15,customers01.csv
4,C0005,Bristol,2023-07-13,customers01.csv
5,C0006,London,2023-08-28,customers01.csv
6,C0007,Leeds,2022-02-02,customers01.csv
7,C0008,London,2022-04-06,customers01.csv
8,C0009,Manchester,2022-08-27,customers01.csv
9,C0010,Liverpool,2023-09-09,customers01.csv


## 3.2 Patients — transformed data

Now apply the transformation logic using a `SELECT` statement.


In [6]:
transformed_customers_query="""
SELECT
    TRIM(customer_id) AS customer_id,
    INITCAP(TRIM(city)) AS city,
    TO_DATE(signup_date, 'YYYY-MM-DD') AS signup_date,
    source_file
FROM raw.customers
WHERE customer_id IS NOT NULL
ORDER BY customer_id
LIMIT 10;
"""

customers_df = run_query(transformed_customers_query)
customers_df.head(10)

,customer_id,city,signup_date,source_file
0,C0001,Bristol,2022-04-25,customers01.csv
1,C0002,London,2022-10-09,customers01.csv
2,C0003,Manchester,2022-08-17,customers01.csv
3,C0004,Manchester,2022-04-15,customers01.csv
4,C0005,Bristol,2023-07-13,customers01.csv
5,C0006,London,2023-08-28,customers01.csv
6,C0007,Leeds,2022-02-02,customers01.csv
7,C0008,London,2022-04-06,customers01.csv
8,C0009,Manchester,2022-08-27,customers01.csv
9,C0010,Liverpool,2023-09-09,customers01.csv
